In [ ]:
# ===============================================
# In This file:
# The Single-generator R-GAN framework is implemented on the MNIST dataset for Targeted Attack (target = class 0), according the R-GAN paper.
# This framework is also compatible with the Fashion-MNIST dataset with changing the dataset to Fashion-MNIST.
# The robustness (accuracy) for classifiers of R-net, C-net, and T-net are evaluated against attacks generated by G-net, FGSM, PGD, and C&W.
# ===============================================

In [ ]:

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


In [ ]:

# ===============================================
# In This Cell:
# The architecture of the classifiers f-net (C-net) and R-net is defined.
# The pretrained classifier C-net described in the paper is referred to as f-net in the code (f-net in code = C-net in the paper).
# ===============================================

class ModelC(nn.Module):
    def __init__(self):
        super(ModelC, self).__init__()
        # --- Convolutional layers ---
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)  # Conv(32,3,3)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)                           # Conv(32,3,3)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)                                 # MaxPooling(2,2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)                           # Conv(64,3,3)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)                           # Conv(64,3,3)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)                                 # MaxPooling(2,2)

        self.fc1 = nn.Linear(64 * 8 * 8, 200)                                              # FC(200)
        self.fc2 = nn.Linear(200, 10)                                                      # FC(10)

    def forward(self, x):
        # --- Block 1 ---
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool1(x)

        # --- Block 2 ---
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)      # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = F.softmax(x, dim=1)        # Softmax for probabilities
        return x


In [ ]:
# ===============================================
# In This Cell:
# The architecture of the generator is defined.
# The generator is denoted as G in the code and as G-net in the paper (G in the code = G-net in the paper).
# ===============================================

class ResnetBlock(nn.Module):
    def __init__(self, dim, norm_layer=nn.BatchNorm2d, use_dropout=False, use_bias=False):
        super(ResnetBlock, self).__init__()
        conv_block = [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim),
            nn.ReLU(True)
        ]
        if use_dropout:
            conv_block += [nn.Dropout(0.5)]
        conv_block += [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim)
        ]
        self.conv_block = nn.Sequential(*conv_block)

    def forward(self, x):
        return x + self.conv_block(x)


class Generator(nn.Module):
    def __init__(self, gen_input_nc, image_nc):
        super(Generator, self).__init__()
        encoder = [
            nn.Conv2d(gen_input_nc, 8, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(8),
            nn.ReLU(),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(32),
            nn.ReLU(),
        ]
        bottle_neck = [ResnetBlock(32), ResnetBlock(32), ResnetBlock(32), ResnetBlock(32)]
        decoder = [
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(16),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(8),
            nn.ReLU(),
            nn.Conv2d(8, image_nc, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        ]
        self.encoder = nn.Sequential(*encoder)
        self.bottle_neck = nn.Sequential(*bottle_neck)
        self.decoder = nn.Sequential(*decoder)

    def forward(self, x):
        
        return self.decoder(self.bottle_neck(self.encoder(x)))


In [ ]:

# ===============================================
# In This Cell:
# The dataset MNIST is loaded.
# For run this file with Fashion-MNIST, load Fashion-MNIST insted of MNIST.
# ===============================================


from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# === Device ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

##########################################################
# Specify the path of the downloaded MNIST dataset:
mnist_path = r"C:\Users\Administrator\Desktop\R-GAN\1- MNIST\Dataset"
###########################################################

# === Transform ===
transform = transforms.ToTensor()

# === Load MNIST from local folder ===
mnist_train = datasets.MNIST(
    root=mnist_path,
    train=True,
    transform=transform,
    download=True
)

mnist_test = datasets.MNIST(
    root=mnist_path,
    train=False,
    transform=transform,
    download=False
)

# === Dataloaders ===
batch_size = 128

train_dataloader = DataLoader(
    mnist_train, batch_size=batch_size, shuffle=True,
    num_workers=2
)

test_dataloader = DataLoader(
    mnist_test, batch_size=batch_size, shuffle=False,
    num_workers=2
)

print("Training samples:", len(mnist_train))
print("Test samples:", len(mnist_test))


Device: cuda
Training samples: 60000
Test samples: 10000


In [ ]:
# ===============================================
# In This Cell:
# The classifier f-net (C-net) is trained.
# ===============================================

f_net = ModelC().to(device)
optimizer_f = torch.optim.Adam(f_net.parameters(), lr=1e-3)

epochs_f = 40

for epoch in range(1, epochs_f+1):
    f_net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in train_dataloader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer_f.zero_grad()
        logits = f_net(imgs)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer_f.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"[f train] Epoch {epoch}/{epochs_f} | Loss: {running_loss/len(train_dataloader):.4f} | Acc: {acc:.2f}%")



# Freeze f_net for later use in R-GAN training
f_net.eval()
for p in f_net.parameters():
    p.requires_grad = False

[f train] Epoch 1/30 | Loss: 0.2142 | Acc: 93.07%
[f train] Epoch 2/30 | Loss: 0.0498 | Acc: 98.47%
[f train] Epoch 3/30 | Loss: 0.0342 | Acc: 98.91%
[f train] Epoch 4/30 | Loss: 0.0238 | Acc: 99.24%
[f train] Epoch 5/30 | Loss: 0.0189 | Acc: 99.39%
[f train] Epoch 6/30 | Loss: 0.0146 | Acc: 99.54%
[f train] Epoch 7/30 | Loss: 0.0142 | Acc: 99.52%
[f train] Epoch 8/30 | Loss: 0.0101 | Acc: 99.69%
[f train] Epoch 9/30 | Loss: 0.0093 | Acc: 99.67%
[f train] Epoch 10/30 | Loss: 0.0086 | Acc: 99.69%
[f train] Epoch 11/30 | Loss: 0.0087 | Acc: 99.70%
[f train] Epoch 12/30 | Loss: 0.0061 | Acc: 99.82%
[f train] Epoch 13/30 | Loss: 0.0049 | Acc: 99.83%
[f train] Epoch 14/30 | Loss: 0.0062 | Acc: 99.80%
[f train] Epoch 15/30 | Loss: 0.0041 | Acc: 99.87%
[f train] Epoch 16/30 | Loss: 0.0044 | Acc: 99.86%
[f train] Epoch 17/30 | Loss: 0.0056 | Acc: 99.84%
[f train] Epoch 18/30 | Loss: 0.0043 | Acc: 99.85%
[f train] Epoch 19/30 | Loss: 0.0039 | Acc: 99.88%
[f train] Epoch 20/30 | Loss: 0.0019 | A

In [ ]:
# ===============================================
# In This Cell:
# The R-GAN Framework is trained.
# R-net and G(G-net) are trained together competitively in the R-GAN Framework.
# The R-GAN Framework in this cell and file is implemented for the Targeted Attack scenario.
# Generator G forces both f-net & R-net classifiers towards the class 0 category (target = class 0).
# ===============================================

import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

R_net = ModelC().to(device)               # R-net classifier that would be trained robust
G     = Generator(1, 1).to(device)        # generator: input 1 channel, output 1 channel

optimizer_R = torch.optim.Adam(R_net.parameters(), lr=1e-4, weight_decay=1e-5)
optimizer_G = torch.optim.Adam(G.parameters(), lr=1e-3)

# -------------------------
#  Pretrained f_net (already loaded)
# -------------------------
f_net.eval()
for p in f_net.parameters():
    p.requires_grad = False

# -------------------------
# Hyperparameters
# -------------------------
eps = 0.3                    # Generated perturbation bounded on per-pixel (L_inf bound)
lambda_f = 5.0               # Coefficient for fooling pretrained f-net during generator G update
lambda_R = 5.0               # Coefficient for fooling R-net during generator G update
lambda_pert = 1.0            # Coefficient for perturbation L2 regularizer during generator G update
target_class = 0             # targeted attack label
num_epochs = 30
print_every = 1


# -------------------------
# TARGET cross-entropy helper
# -------------------------
def targeted_ce_loss(logits, target_label):
    """
    Minimize CE(logits, target)  →  push model output toward target class.
    """
    target = torch.full((logits.size(0),), target_label, dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, target)


# ============================================================
#                   TRAINING LOOP
# ============================================================

for epoch in range(1, num_epochs + 1):

    G.train()
    R_net.train()

    total_loss_R = 0
    total_loss_G = 0
    total_loss_f = 0
    total_loss_pert = 0
    batches = 0

    # ----------------------------------------
    # TRAINING OVER ALL MINI-BATCHES
    # ----------------------------------------
    for imgs, labels in train_dataloader:
        imgs   = imgs.to(device)
        labels = labels.to(device)
        batches += 1

        # ==============================================
        # 1) R-NET UPDATE (G frozen)
        # ==============================================
        G.eval()
        with torch.no_grad():
            delta = torch.clamp(G(imgs), -eps, eps)
            adv_imgs = torch.clamp(imgs + delta, 0.0, 1.0)

        R_net.train()
        logits_clean = R_net(imgs)
        logits_adv   = R_net(adv_imgs)

        # R_net must classify correctly (true labels)
        loss_R = F.cross_entropy(logits_clean, labels) + \
                 F.cross_entropy(logits_adv, labels)

        optimizer_R.zero_grad()
        loss_R.backward()
        optimizer_R.step()

        total_loss_R += loss_R.item()

        # ==============================================
        # 2) GENERATOR G UPDATE (Freeze f_net & R_net)
        # ==============================================
        G.train()
        R_net.eval()
        for p in R_net.parameters():
            p.requires_grad = False

        delta = torch.clamp(G(imgs), -eps, eps)
        adv_imgs = torch.clamp(imgs + delta, 0.0, 1.0)

        # --- targeted losses ---
        # pushing logits toward class 0
        loss_f_target = targeted_ce_loss(f_net(adv_imgs), target_class)
        loss_R_target = targeted_ce_loss(R_net(adv_imgs), target_class)

        # L2 regularization
        loss_pert = torch.mean(torch.norm(delta.view(delta.size(0), -1), dim=1, p=2))

        # Full generator objective
        loss_G = lambda_f * loss_f_target + \
                 lambda_R * loss_R_target + \
                 lambda_pert * loss_pert

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

        total_loss_G += loss_G.item()
        total_loss_f += loss_f_target.item()
        total_loss_pert += loss_pert.item()

        for p in R_net.parameters():
            p.requires_grad = True


    if epoch % print_every == 0:
        print(f"[Epoch {epoch}/{num_epochs}] "
              f"Loss_R: {total_loss_R/batches:.4f} | "
              f"Loss_G: {total_loss_G/batches:.4f} | "
              f"Loss_f_target: {total_loss_f/batches:.4f} | "
              f"Loss_pert(L2): {total_loss_pert/batches:.4f}")


    # QUICK TEST EVAL EVERY EPOCH
    G.eval()
    R_net.eval()
    f_net.eval()

    correct_clean_R = correct_adv_R = 0
    correct_clean_f = correct_adv_f = 0
    total_test = 0

    with torch.no_grad():
        for imgs_t, labels_t in test_dataloader:
            imgs_t   = imgs_t.to(device)
            labels_t = labels_t.to(device)
            total_test += labels_t.size(0)

            # clean predictions
            correct_clean_R += (R_net(imgs_t).argmax(dim=1) == labels_t).sum().item()
            correct_clean_f += (f_net(imgs_t).argmax(dim=1) == labels_t).sum().item()

            # adv predictions
            delta_t = torch.clamp(G(imgs_t), -eps, eps)
            adv_t = torch.clamp(imgs_t + delta_t, 0.0, 1.0)

            correct_adv_R += (R_net(adv_t).argmax(dim=1) == labels_t).sum().item()
            correct_adv_f += (f_net(adv_t).argmax(dim=1) == labels_t).sum().item()

    print(f" Eval - R clean acc: {100*correct_clean_R/total_test:.2f}% | "
          f"R adv acc: {100*correct_adv_R/total_test:.2f}%")
    print(f"      f clean acc: {100*correct_clean_f/total_test:.2f}% | "
          f"f adv acc: {100*correct_adv_f/total_test:.2f}%")



Device: cuda
[Epoch 1/30] Loss_R: 1.4210 | Loss_G: 40.7611 | Loss_f_target: 1.7164 | Loss_pert(L2): 6.5689
 Eval - R clean acc: 93.09% | R adv acc: 89.71%
      f clean acc: 99.20% | f adv acc: 20.06%
[Epoch 2/30] Loss_R: 0.4751 | Loss_G: 45.1498 | Loss_f_target: 0.5536 | Loss_pert(L2): 5.6620
 Eval - R clean acc: 96.63% | R adv acc: 92.49%
      f clean acc: 99.20% | f adv acc: 18.49%
[Epoch 3/30] Loss_R: 0.3399 | Loss_G: 48.7055 | Loss_f_target: 0.4259 | Loss_pert(L2): 5.2947
 Eval - R clean acc: 97.38% | R adv acc: 94.94%
      f clean acc: 99.20% | f adv acc: 16.28%
[Epoch 4/30] Loss_R: 0.2716 | Loss_G: 52.2436 | Loss_f_target: 0.3588 | Loss_pert(L2): 5.1968
 Eval - R clean acc: 97.82% | R adv acc: 94.24%
      f clean acc: 99.20% | f adv acc: 15.58%
[Epoch 5/30] Loss_R: 0.2253 | Loss_G: 55.5239 | Loss_f_target: 0.3657 | Loss_pert(L2): 5.0863
 Eval - R clean acc: 98.05% | R adv acc: 96.02%
      f clean acc: 99.20% | f adv acc: 17.31%
[Epoch 6/30] Loss_R: 0.1896 | Loss_G: 58.2802 |

In [ ]:
# ===============================================
# In the three cells below:
# T-net is trined using Model A.
# To train T-net with Model B or Model C, as described in the R-GAN Paper, set T-net according to the architecture of these models.
# The Architecture of Model B or C is defined in the "Architecture of Models" file.
# ===============================================

In [ ]:

train_dataloader = DataLoader(
    mnist_train, batch_size=batch_size, shuffle=True,
    num_workers=2
)

In [ ]:
# ===============================================
# In This Cell:
# The Architecture of Model A (T-net) is defined.
# ===============================================

class ModelA(nn.Module):
    def __init__(self):
        super(ModelA, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, stride=1, padding=2)

        # Dropout layers
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Convolutional block
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.dropout1(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected block
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)

        return x


In [ ]:
# ===============================================
# In This Cell:
# The classifier of T-net is trained.
# ===============================================

T_net = ModelA().to(device)
optimizer_T = torch.optim.Adam(T_net.parameters(), lr=1e-3)

epochs_T = 40  

for epoch in range(1, epochs_T+1):
    T_net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in train_dataloader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer_T.zero_grad()
        logits = T_net(imgs)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer_T.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"[T train] Epoch {epoch}/{epochs_T} | Loss: {running_loss/len(train_dataloader):.4f} | Acc: {acc:.2f}%")



# Freeze f_net for later use in R-GAN training
T_net.eval()
for p in T_net.parameters():
    p.requires_grad = False

[T train] Epoch 1/30 | Loss: 0.2050 | Acc: 93.82%
[T train] Epoch 2/30 | Loss: 0.0737 | Acc: 97.84%
[T train] Epoch 3/30 | Loss: 0.0525 | Acc: 98.41%
[T train] Epoch 4/30 | Loss: 0.0418 | Acc: 98.71%
[T train] Epoch 5/30 | Loss: 0.0346 | Acc: 98.97%
[T train] Epoch 6/30 | Loss: 0.0294 | Acc: 99.07%
[T train] Epoch 7/30 | Loss: 0.0243 | Acc: 99.23%
[T train] Epoch 8/30 | Loss: 0.0201 | Acc: 99.36%
[T train] Epoch 9/30 | Loss: 0.0203 | Acc: 99.33%
[T train] Epoch 10/30 | Loss: 0.0181 | Acc: 99.41%
[T train] Epoch 11/30 | Loss: 0.0145 | Acc: 99.54%
[T train] Epoch 12/30 | Loss: 0.0136 | Acc: 99.56%
[T train] Epoch 13/30 | Loss: 0.0147 | Acc: 99.49%
[T train] Epoch 14/30 | Loss: 0.0114 | Acc: 99.62%
[T train] Epoch 15/30 | Loss: 0.0115 | Acc: 99.61%
[T train] Epoch 16/30 | Loss: 0.0099 | Acc: 99.68%
[T train] Epoch 17/30 | Loss: 0.0100 | Acc: 99.66%
[T train] Epoch 18/30 | Loss: 0.0097 | Acc: 99.69%
[T train] Epoch 19/30 | Loss: 0.0094 | Acc: 99.69%
[T train] Epoch 20/30 | Loss: 0.0087 | A

In [ ]:
#================================================================================================================
### Evaluation: Clean and Generator-perturbed  accuracy toward class "0" (target = class 0) for T_net, R_net, f_net ###
# ================================================================================================================


In [ ]:
# ================================================
# In This Cell: 
# The accuracy of T_net, R_net, and f_net for perturbed data by generator G (G-net attacker) and clean data is evaluated.
# In this file, the generator G-net produces Targeted Attacks toward class 0 category (target = class 0).
# ================================================


eps = 0.3
# Put all networks in evaluation mode
f_net.eval()
R_net.eval()
T_net.eval()
G.eval()

correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

total = 0

print("Running evaluation on MNIST test set...")

with torch.no_grad():
    for imgs, labels in test_dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        total += labels.size(0)

        # ----------------------------
        # 1) Clean Accuracy
        # ----------------------------
        preds_f_clean = f_net(imgs).argmax(dim=1)
        preds_R_clean = R_net(imgs).argmax(dim=1)
        preds_T_clean = T_net(imgs).argmax(dim=1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

        # ----------------------------
        # 2) Generator-perturbed data
        # ----------------------------
        perturb = G(imgs)
        perturb = torch.clamp(perturb, -eps, eps)  # L_inf bound
        adv_imgs = torch.clamp(imgs + perturb, 0.0, 1.0)

        preds_f_adv = f_net(adv_imgs).argmax(dim=1)
        preds_R_adv = R_net(adv_imgs).argmax(dim=1)
        preds_T_adv = T_net(adv_imgs).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()


# =============================
# Final Accuracy Results
# =============================
clean_acc_f = 100 * correct_clean_f / total
adv_acc_f   = 100 * correct_adv_f / total

clean_acc_R = 100 * correct_clean_R / total
adv_acc_R   = 100 * correct_adv_R / total

clean_acc_T = 100 * correct_clean_T / total
adv_acc_T   = 100 * correct_adv_T / total

print("\n=========== FINAL RESULTS ===========")
print(f"f_net  | Clean: {clean_acc_f:.2f}%   | Adv (G): {adv_acc_f:.2f}%")
print(f"R_net  | Clean: {clean_acc_R:.2f}%   | Adv (G): {adv_acc_R:.2f}%")
print(f"T_net  | Clean: {clean_acc_T:.2f}%   | Adv (G): {adv_acc_T:.2f}%")
print("=====================================\n")

Running evaluation on MNIST test set...

=========== FINAL RESULTS ===========
f_net  | Clean: 99.20%   | Adv (G): 24.67%
R_net  | Clean: 99.11%   | Adv (G): 97.68%
T_net  | Clean: 99.29%   | Adv (G): 65.54%



In [ ]:
#================================================================
# In the The Following Cells:
# The accuracy of R-net and T-net against Targeted Transferable Attacks crafted on f-net (C-net) using FGSM, PGD, and C&W attacks is evaluated.
# The Targeted attacks are toward class 0 (target = class 0).
# ===============================================================


In [ ]:
# ===============================================
# In This Cell:
# Targeted FGSM attacks (target = class 0) crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against targeted FGSM attacks crafted on f_net.
# ===============================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval()
R_net.eval()
T_net.eval()

eps = 0.3   # MNIST L_inf epsilon
target_class = 0

total = 0

correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

# How many samples moved TOWARD target on f_net?
target_hit_f = 0
transfer_R = 0
transfer_T = 0

print("Running TARGETED FGSM crafted on f_net (target = 0) ...")

for imgs, labels in tqdm(test_dataloader):

    imgs = imgs.to(device)
    labels = labels.to(device)
    total += labels.size(0)

    # --------------------
    # CLEAN predictions
    # --------------------
    with torch.no_grad():
        preds_f_clean = f_net(imgs).argmax(dim=1)
        preds_R_clean = R_net(imgs).argmax(dim=1)
        preds_T_clean = T_net(imgs).argmax(dim=1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # --------------------
    # TARGETED FGSM on f_net
    # --------------------
    target_labels = torch.full_like(labels, target_class).to(device)

    imgs_adv = imgs.clone().detach().requires_grad_(True)
    logits = f_net(imgs_adv)
    loss = F.cross_entropy(logits, target_labels)
    loss.backward()

    # Targeted FGSM (subtract gradient)
    tg_fgsm = imgs_adv - eps * imgs_adv.grad.sign()
    adv_imgs = torch.clamp(tg_fgsm, 0, 1).detach()

    # --------------------
    # ADV predictions
    # --------------------
    with torch.no_grad():
        preds_f_adv = f_net(adv_imgs).argmax(dim=1)
        preds_R_adv = R_net(adv_imgs).argmax(dim=1)
        preds_T_adv = T_net(adv_imgs).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        # Did f_net hit the target class?
        mask_hit_f = (preds_f_adv == target_class)
        num_target_f = mask_hit_f.sum().item()
        target_hit_f += num_target_f

        if num_target_f > 0:
            transfer_R += ((preds_R_adv == target_class) & mask_hit_f).sum().item()
            transfer_T += ((preds_T_adv == target_class) & mask_hit_f).sum().item()


# ==========================
# Compute Final Accuracies
# ==========================

clean_acc_f = 100 * correct_clean_f / total
clean_acc_R = 100 * correct_clean_R / total
clean_acc_T = 100 * correct_clean_T / total

adv_acc_f = 100 * correct_adv_f / total
adv_acc_R = 100 * correct_adv_R / total
adv_acc_T = 100 * correct_adv_T / total

transfer_R_rate = 100 * transfer_R / target_hit_f if target_hit_f > 0 else 0
transfer_T_rate = 100 * transfer_T / target_hit_f if target_hit_f > 0 else 0


# ==========================
# PRINT RESULTS
# ==========================

print("\n=== TARGETED FGSM crafted on f_net (target = 0) ===")
print(f"Total samples: {total}")

print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")

print(f"\nAdversarial accuracy (adv crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}%")
print(f"  R_net: {adv_acc_R:.2f}%")
print(f"  T_net: {adv_acc_T:.2f}%")

print(f"\nf_net reached target class (0) for {target_hit_f}/{total} samples "
      f"({100 * target_hit_f / total:.2f}%)")

print(f"Transfer success for target=0 → R_net: {transfer_R_rate:.2f}%")
print(f"Transfer success for target=0 → T_net: {transfer_T_rate:.2f}%")


Running TARGETED FGSM crafted on f_net (target = 0) ...


100%|██████████████████████████████████████████████████████████████████████████████████| 79/79 [00:06<00:00, 11.96it/s]


=== TARGETED FGSM crafted on f_net (target = 0) ===
Total samples: 10000

Clean accuracy:
  f_net: 99.20%
  R_net: 99.11%
  T_net: 99.38%

Adversarial accuracy (adv crafted for f_net):
  f_net: 44.13%
  R_net: 85.70%
  T_net: 39.40%

f_net reached target class (0) for 1329/10000 samples (13.29%)
Transfer success for target=0 → R_net: 25.36%
Transfer success for target=0 → T_net: 69.07%


In [ ]:

# ===============================================
# In This Cell:
# Targeted PGD attacks (target = class 0) crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against Targeted PGD attacks crafted on f_net.
# ===============================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval()
R_net.eval()
T_net.eval()

eps = 0.25
pgd_steps = 30
step_size = 0.01

total = 0

correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

f_fool_count = 0
transfer_R_on_f = 0
transfer_T_on_f = 0

print("Running TARGETED PGD crafted on f_net (target = 0)...")

for imgs, labels in tqdm(test_dataloader):

    imgs = imgs.to(device)
    labels = labels.to(device)
    total += labels.size(0)

    target_labels = torch.zeros_like(labels).to(device)   # TARGET CLASS = 0

    # ----------------------------
    # CLEAN predictions
    # ----------------------------
    with torch.no_grad():
        preds_f_clean = f_net(imgs).argmax(1)
        preds_R_clean = R_net(imgs).argmax(1)
        preds_T_clean = T_net(imgs).argmax(1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # ----------------------------
    # TARGETED PGD ON f_net
    # ----------------------------
    adv = imgs.clone().detach()

    for _ in range(pgd_steps):
        adv.requires_grad_(True)
        logits = f_net(adv)

        loss = -F.cross_entropy(logits, target_labels)
        loss.backward()

        with torch.no_grad():
            adv = adv + step_size * adv.grad.sign()  # targeted PGD = gradient ASCENT on negative CE
            adv = torch.max(torch.min(adv, imgs + eps), imgs - eps)
            adv = torch.clamp(adv, 0, 1)

    adv_imgs = adv.detach()

    # ----------------------------
    # ADV predictions
    # ----------------------------
    with torch.no_grad():
        preds_f_adv = f_net(adv_imgs).argmax(1)
        preds_R_adv = R_net(adv_imgs).argmax(1)
        preds_T_adv = T_net(adv_imgs).argmax(1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        # Fooling means: model predicts TARGET class (0), not the true label
        f_fool_mask = (preds_f_adv == 0) & (labels != 0)
        num_f_fool = f_fool_mask.sum().item()
        f_fool_count += num_f_fool

        if num_f_fool > 0:
            transfer_R_on_f += ((preds_R_adv == 0) & f_fool_mask).sum().item()
            transfer_T_on_f += ((preds_T_adv == 0) & f_fool_mask).sum().item()


# ======================
# RESULTS
# ======================
print("\n=== TARGETED PGD (crafted on f_net → target class = 0) ===")
print(f"Total test samples: {total}")

print("\nClean accuracy:")
print(f"  f_net: {100*correct_clean_f/total:.2f}%")
print(f"  R_net: {100*correct_clean_R/total:.2f}%")
print(f"  T_net: {100*correct_clean_T/total:.2f}%")

print("\nAdv accuracy:")
print(f"  f_net: {100*correct_adv_f/total:.2f}%")
print(f"  R_net: {100*correct_adv_R/total:.2f}%")
print(f"  T_net: {100*correct_adv_T/total:.2f}%")

print(f"\nTargeted f_net → 0 fooled count: {f_fool_count} / {total}")
print(f"Transfer to R_net: {100*transfer_R_on_f/max(f_fool_count,1):.2f}%")
print(f"Transfer to T_net: {100*transfer_T_on_f/max(f_fool_count,1):.2f}%")


Running TARGETED PGD crafted on f_net (target = 0)...


100%|██████████████████████████████████████████████████████████████████████████████████| 79/79 [00:11<00:00,  6.93it/s]


=== TARGETED PGD (crafted on f_net → target class = 0) ===
Total test samples: 10000

Clean accuracy:
  f_net: 99.20%
  R_net: 99.11%
  T_net: 99.38%

Adv accuracy:
  f_net: 12.29%
  R_net: 98.42%
  T_net: 67.43%

Targeted f_net → 0 fooled count: 8475 / 10000
Transfer to R_net: 0.00%
Transfer to T_net: 9.90%


In [ ]:

# ===============================================
# In This Cell:
# Targeted C&W attacks (target = class 0) crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against Targeted C&W attacks crafted on f_net.
# ===============================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval()
R_net.eval()
T_net.eval()

target_class = 0
cw_steps = 1000         # optimization steps
lr = 0.01               # Adam learning rate
c = 0.1                 # weight on L2 term
kappa = 0               # confidence margin
eps = 0.3               # optional clamp on perturbation (L_inf)

total = 0

correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

f_fool_count = 0
transfer_R = 0
transfer_T = 0

print("Running TARGETED C&W L2 crafted on f_net (target = 0)...")

for imgs, labels in tqdm(test_dataloader):

    imgs = imgs.to(device)
    labels = labels.to(device)
    total += labels.size(0)

    target_labels = torch.zeros_like(labels).to(device)   # TARGET CLASS = 0

    # ----------------------------
    # CLEAN predictions
    # ----------------------------
    with torch.no_grad():
        preds_f_clean = f_net(imgs).argmax(1)
        preds_R_clean = R_net(imgs).argmax(1)
        preds_T_clean = T_net(imgs).argmax(1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # ==================================================
    #       TARGETED C&W: Optimize perturbation 
    # ==================================================
    imgs_orig = imgs.clone().detach()
    delta = torch.zeros_like(imgs, requires_grad=True)
    optimizer = torch.optim.Adam([delta], lr=lr)

    for _ in range(cw_steps):
        optimizer.zero_grad()

        adv = torch.clamp(imgs_orig + delta, 0.0, 1.0)

        logits = f_net(adv)
        tgt_logits = logits[torch.arange(labels.size(0)), target_labels]
        other_logits = logits.clone()
        other_logits[torch.arange(labels.size(0)), target_labels] = -1e9
        max_other, _ = other_logits.max(dim=1)

        # targeted hinge loss:
        # want: target_logit >= max_other + kappa
        hinge = torch.clamp(max_other - tgt_logits + kappa, min=0)

        l2 = torch.sum(delta.view(delta.size(0), -1)**2, dim=1)
        loss = hinge.mean() + c * l2.mean()

        loss.backward()
        optimizer.step()

        # optional: clamp to L_inf bound
        delta.data = torch.clamp(delta.data, -eps, eps)

    adv_imgs = torch.clamp(imgs_orig + delta.detach(), 0.0, 1.0)

    # ----------------------------
    # ADV predictions
    # ----------------------------
    with torch.no_grad():
        preds_f_adv = f_net(adv_imgs).argmax(1)
        preds_R_adv = R_net(adv_imgs).argmax(1)
        preds_T_adv = T_net(adv_imgs).argmax(1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        # Fooling means: model predicts TARGET class (0)
        f_fool_mask = (preds_f_adv == target_class) & (labels != target_class)
        num_f_fool = f_fool_mask.sum().item()
        f_fool_count += num_f_fool

        if num_f_fool > 0:
            transfer_R += ((preds_R_adv == target_class) & f_fool_mask).sum().item()
            transfer_T += ((preds_T_adv == target_class) & f_fool_mask).sum().item()


# ======================
# RESULTS
# ======================
print("\n=== TARGETED C&W L2 (crafted on f_net → target = 0) ===")
print(f"Total test samples: {total}")

print("\nClean accuracy:")
print(f"  f_net: {100*correct_clean_f/total:.2f}%")
print(f"  R_net: {100*correct_clean_R/total:.2f}%")
print(f"  T_net: {100*correct_clean_T/total:.2f}%")

print("\nAdversarial (C&W) accuracy:")
print(f"  f_net: {100*correct_adv_f/total:.2f}%")
print(f"  R_net: {100*correct_adv_R/total:.2f}%")
print(f"  T_net: {100*correct_adv_T/total:.2f}%")

print(f"\nTargeted f_net → 0 success count: {f_fool_count} / {total}")
print(f"Transfer to R_net: {100*transfer_R/max(f_fool_count,1):.2f}%")
print(f"Transfer to T_net: {100*transfer_T/max(f_fool_count,1):.2f}%")


Running TARGETED C&W L2 crafted on f_net (target = 0)...


100%|██████████████████████████████████████████████████████████████████████████████████| 79/79 [01:34<00:00,  1.19s/it]


=== TARGETED C&W L2 (crafted on f_net → target = 0) ===
Total test samples: 10000

Clean accuracy:
  f_net: 99.20%
  R_net: 99.11%
  T_net: 99.38%

Adversarial (C&W) accuracy:
  f_net: 11.54%
  R_net: 98.89%
  T_net: 89.74%

Targeted f_net → 0 success count: 8753 / 10000
Transfer to R_net: 0.03%
Transfer to T_net: 1.50%
